<a href="https://colab.research.google.com/github/DeepLabCut/DeepLabCut/blob/master/examples/COLAB/COLAB_DEMO_mouse_openfield.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 中打开"/></a>

# 单只小鼠数据演示 DeepLabCut

一些有用的链接：

- [DeepLabCut 的 GitHub：github.com/DeepLabCut/DeepLabCut](https://github.com/DeepLabCut/DeepLabCut)
- [DeepLabCut 的文档：单动物项目用户指南](https://deeplabcut.github.io/DeepLabCut/docs/standardDeepLabCut_UserGuide.html)

![alt text](https://images.squarespace-cdn.com/content/v1/57f6d51c9f74566f55ecf271/1559935526258-KFYZC8BDHK01ZIDPNVIX/mouse_skel_trail.gif?format=450w)

演示支持：Nath\*, Mathis\* 等人。*使用 DeepLabCut 进行跨物种行为期间的无标记 3D 姿态估计。Nature Protocols, 2019*

本 Notebook 演示了使用 DeepLabCut 处理我们的演示数据的必要步骤。我们提供了来自 Mathis 等人（2018 年发表于 Nature Neuroscience）的小鼠数据的子集。

本演示 Notebook 主要展示了训练和评估模型的**最简单**的代码，但许多函数都具有额外的功能，因此请务必查阅概述和协议论文！

本 Notebook 说明了如何使用云端执行以下操作：

- 加载演示数据
- 创建训练集
- 训练网络
- 评估网络
- 分析新的视频

## 安装

### 首先，进入 “Runtime”（运行时） -> “Change runtime type”（更改运行时类型） -> 选择 “Python3”，然后选择 “GPU”

In [ ]:
# Clone the entire deeplabcut repo so we can use the demo data:
!git clone -l -s https://github.com/DeepLabCut/DeepLabCut.git cloned-DLC-repo
%cd cloned-DLC-repo
!ls

In [ ]:
%cd /content/cloned-DLC-repo/examples/openfield-Pranav-2018-10-30
!ls

In [ ]:
# Install the latest DeepLabCut version (this will take a few minutes to install all the dependencies!)
%cd /content/cloned-DLC-repo/
%pip install "."

### 请注意，在继续之前，请点击上方输出中的“重启运行时”！

In [ ]:
import deeplabcut

In [ ]:
# Create a path variable that links to the config file:
path_config_file = '/content/cloned-DLC-repo/examples/openfield-Pranav-2018-10-30/config.yaml'

# Loading example data set:
deeplabcut.load_demo_data(path_config_file)

# Automatically update some hyperparameters for training, 
# here rotations to +/- 180 degrees. This can be helpful for optimizing performance. 
# see Primer -- Mathis et al. Neuron 2020
from deeplabcut.core.config import read_config_as_dict
import deeplabcut.pose_estimation_pytorch as dlc_torch

loader = dlc_torch.DLCLoader(
    config=path_config_file,  
    trainset_index=0,
    shuffle=1,
)

# Get the pytorch config path 
pytorch_config_path = loader.model_folder / "pytorch_config.yaml"

model_cfg = read_config_as_dict(pytorch_config_path)
model_cfg['data']["train"]["affine"]["rotation"]=180

# Save the modified config
dlc_torch.config.write_config(pytorch_config_path,model_cfg)

## 开始训练：
此函数针对训练数据集的特定洗牌（shuffle）来训练网络模型。

In [ ]:
# Let's also change the display and save_epochs just in case Colab takes away
# the GPU... If that happens, you can reload from a saved point using the
# `snapshot_path` argument to `deeplabcut.train_network`:
#   deeplabcut.train_network(..., snapshot_path="/content/.../snapshot-050.pt")

# Typically, you want to train to ~200 epochs. We set the batch size to 8 to
# utilize the GPU's capabilities.

# More info and there are more things you can set:
#   https://deeplabcut.github.io/DeepLabCut/docs/standardDeepLabCut_UserGuide.html#g-train-the-network

deeplabcut.train_network(
    path_config_file,
    shuffle=1,
    save_epochs=5,
    epochs=200,
    batch_size=8,
)

# This will run until you stop it (CTRL+C), or hit "STOP" icon, or when it hits the end.

我们建议您运行大约 100 个 epoch，这只是一个演示。这大约需要 15 分钟。请注意，**当您点击“停止”按钮时，您会收到一个 `KeyboardInterrupt` “错误”！不必担心！ :)**

一个新的快照（snapshot）每经过 `save_epochs` 个周期就会保存一次。因此，一旦您达到 80 个 epoch，您在 `/content/cloned-DLC-repo/examples/openfield-Pranav-2018-10-30/dlc-models-pytorch/iteration-0/openfieldOct30-trainset95shuffle1/train` 中的最新快照应该是 `snapshot-80.pt`。在训练过程中评估出的最佳快照也会被保存，其命名格式为 `snapshot-best-XX.pt`，其中 `XX` 是模型训练所经历的完整 epoch 数量。

## 开始评估：
此函数用于对特定轮换（shuffle）或所有轮次（shuffles）在特定状态或所有状态下的训练模型进行评估，评估数据是数据集（images），并将结果作为 .csv 文件存储在 **evaluation-results** 目录下的一个子目录中。

In [ ]:
deeplabcut.evaluate_network(path_config_file, plotting=True)

# Here you want to see a low pixel error! Of course, it can only be as
# good as the labeler, so be sure your labels are good!

**检查图像**：

你可以查看新创建的 `"evaluation-results-pytorch"` 文件夹中的图像。在大约 100 个周期（epochs）时，误差约为 3 像素（但这可能会因你的演示数据是如何划分进行训练而有所不同）。

## 开始分析视频：

此函数用于分析新的视频。用户可以从评估结果中选择最佳模型，并在 `config.yaml` 文件中为变量 **snapshotindex** 指定正确的快照索引。如果未指定，则默认使用最新的快照来分析视频。

分析结果将存储在与视频位于同一目录下的 hd5 文件中。

**对于演示数据，这大约需要 90 秒左右！**（演示帧尺寸为 640x480，在 Google 提供的 T4 GPU 上运行时速度应约为 25 FPS）

In [ ]:
# Enter the list of videos to analyze.
videofile_path = ["/content/cloned-DLC-repo/examples/openfield-Pranav-2018-10-30/videos/m3v1mp4.mp4"]
deeplabcut.analyze_videos(path_config_file, videofile_path, videotype=".mp4")

## 创建带标签的视频：

此函数用于可视化目的，可用于创建 `.mp4` 格式的视频，视频中包含网络预测的标签。此视频将保存在与原始视频相同的目录下。在演示视频上运行时，其速度应约为 215 FPS！

In [ ]:
deeplabcut.create_labeled_video(path_config_file, videofile_path)

## 绘制分析视频的轨迹：
此函数会绘制整个视频中所有身体部位的轨迹。每个身体部位都由一个独特的颜色进行标识。

In [ ]:
deeplabcut.plot_trajectories(path_config_file, videofile_path)